# Tunable compliance — emergent grasps

## In-hand manipulation

The hand closes to a PC1 grasp and three stiffness configurations are applied:
**Uniform** (all fingers at $K_\mathrm{uniform}$),
**Asym 1** (Side 1 = pinky+ring at $K_\mathrm{low}$, Side 2 = thumb+index+middle at $K_\mathrm{high}$),
**Asym 2** (sides swapped).

$\Delta p$ is the fingertip displacement from the PC1 reference position (the target closed-hand configuration).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('../..'))
plt.style.use(os.path.join('../..', 'plot_config.mplstyle'))

FINGERTIPS   = ['thumb', 'index', 'middle', 'ring', 'pinky']
PHASES       = ['uniform', 'asym_a', 'asym_b']
PHASE_LABELS = {'uniform': 'Uniform', 'asym_a': 'Asym 1', 'asym_b': 'Asym 2'}
PHASE_COLORS = {'uniform': '#56B4E9', 'asym_a': '#E69F00', 'asym_b': '#D55E00'}

SIDE_1 = ['pinky', 'ring']             # soft in Asym 1, stiff in Asym 2
SIDE_2 = ['thumb', 'index', 'middle']  # stiff in Asym 1, soft in Asym 2

OUTPUT_INHAND = os.path.join('outputs', 'inhand_manipulation')
os.makedirs(OUTPUT_INHAND, exist_ok=True)


def load_inhand():
    path = Path(OUTPUT_INHAND) / 'inhand_run.csv'
    return pd.read_csv(path) if path.exists() else None


df = load_inhand()
if df is None:
    print('No data — run inhand_manipulation.py first.')
else:
    print(df['phase'].value_counts().to_dict())

### Tip displacement per finger

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(FINGERTIPS))
width = 0.25

for i, phase in enumerate(PHASES):
    means, stds = [], []
    for f in FINGERTIPS:
        if df is not None:
            rows = df[df['phase'] == phase]
            mag  = np.sqrt(rows[f'disp_{f}_x_m']**2 +
                           rows[f'disp_{f}_y_m']**2 +
                           rows[f'disp_{f}_z_m']**2) * 1e3
            means.append(float(mag.mean()))
            stds.append(float(mag.std()))
        else:
            means.append(0.0); stds.append(0.0)
    ax.bar(x + i * width, means, width, yerr=stds, capsize=3,
           label=PHASE_LABELS[phase], color=PHASE_COLORS[phase])

ax.set_xticks(x + width)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel(r'$\|\Delta p\|$ [mm]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_tip_displacement.pdf'), bbox_inches='tight')
plt.show()

### Side 1 vs Side 2 displacement

In [ ]:
def side_stats(df_, phase, fingers):
    if df_ is None:
        return 0.0, 0.0
    rows = df_[df_['phase'] == phase]
    mags = np.concatenate([
        np.sqrt(rows[f'disp_{f}_x_m']**2 +
                rows[f'disp_{f}_y_m']**2 +
                rows[f'disp_{f}_z_m']**2).values * 1e3
        for f in fingers
    ])
    return float(mags.mean()), float(mags.std())


fig, axes = plt.subplots(1, 2, sharey=True)

for ax, phase in zip(axes, ['asym_a', 'asym_b']):
    soft_side  = SIDE_1 if phase == 'asym_a' else SIDE_2
    stiff_side = SIDE_2 if phase == 'asym_a' else SIDE_1
    s1_label   = 'Side 1\n(pinky+ring)'      if phase == 'asym_a' else 'Side 1\n(pinky+ring)'
    s2_label   = 'Side 2\n(thumb+idx+mid)'   if phase == 'asym_a' else 'Side 2\n(thumb+idx+mid)'

    m1, s1 = side_stats(df, phase, SIDE_1)
    m2, s2 = side_stats(df, phase, SIDE_2)
    mu_uni_1, _ = side_stats(df, 'uniform', SIDE_1)
    mu_uni_2, _ = side_stats(df, 'uniform', SIDE_2)

    ax.bar(['Side 1', 'Side 2'], [m1, m2], yerr=[s1, s2], capsize=4,
           color=[PHASE_COLORS[phase], '#AAAAAA'])
    ax.axhline(mu_uni_1, color=PHASE_COLORS['uniform'], ls='--', lw=1.2, alpha=0.8)
    ax.axhline(mu_uni_2, color=PHASE_COLORS['uniform'], ls=':',  lw=1.2, alpha=0.8)
    ax.set_xlabel(PHASE_LABELS[phase])
    if ax is axes[0]:
        ax.set_ylabel(r'$\|\Delta p\|$ [mm]')

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_seesaw.pdf'), bbox_inches='tight')
plt.show()

### Contact force magnitude per finger

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(FINGERTIPS))
width = 0.25

for i, phase in enumerate(PHASES):
    means, stds = [], []
    for f in FINGERTIPS:
        col = f'force_1st_{f}_mag_N'
        if df is not None and col in df.columns:
            rows = df[df['phase'] == phase]
            means.append(float(rows[col].mean()))
            stds.append(float(rows[col].std()))
        else:
            means.append(0.0); stds.append(0.0)
    ax.bar(x + i * width, means, width, yerr=stds, capsize=3,
           label=PHASE_LABELS[phase], color=PHASE_COLORS[phase])

ax.set_xticks(x + width)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel(r'$|F_{\mathrm{tip}}|$ [N]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_force.pdf'), bbox_inches='tight')
plt.show()

### Tip stiffness eigenvalues

In [ ]:
eig_ok = df is not None and 'stiff_1st_thumb_eig_0_Npm' in df.columns

if eig_ok:
    fig, axes = plt.subplots(1, len(FINGERTIPS), figsize=(14, 4), sharey=True)

    for ax, finger in zip(axes, FINGERTIPS):
        data, tick_labels, tick_colors = [], [], []
        for phase in PHASES:
            mask = df['phase'] == phase
            for k in range(3):
                col = f'stiff_1st_{finger}_eig_{k}_Npm'
                data.append(df.loc[mask, col].dropna().values)
                tick_labels.append(f'{PHASE_LABELS[phase]}\n$\\lambda_{k+1}$')
                tick_colors.append(PHASE_COLORS[phase])

        bp = ax.boxplot(data, labels=tick_labels, patch_artist=True,
                        showfliers=False, widths=0.6)
        for patch, color in zip(bp['boxes'], tick_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_xlabel(finger)
        ax.tick_params(axis='x', labelsize=6)
        ax.grid(True, axis='y', alpha=0.3)
        if ax is axes[0]:
            ax.set_ylabel('Eigenvalue [N/m]')

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_stiffness_eigenvalues.pdf'), bbox_inches='tight')
    plt.show()
else:
    print('Stiffness eigenvalue columns not found.')

## Dynamic grasping

The UR5 slides the hand along +X. The hand closes to PC1 at a fixed distance under three stiffness schedules:
**Soft** ($K_\mathrm{soft}$ throughout), **Stiff** ($K_\mathrm{stiff}$ throughout),
**Adaptive** ($K_\mathrm{soft}$ at close, then ramps to $K_\mathrm{stiff}$).

In [ ]:
OUTPUT_GRASP = os.path.join('outputs', 'dynamic_grasp')
os.makedirs(OUTPUT_GRASP, exist_ok=True)

CONDITIONS = ['soft', 'stiff', 'adaptive']
COLORS_DG  = {'soft': '#56B4E9', 'stiff': '#D55E00', 'adaptive': '#009E73'}


def load_dynamic(condition):
    path = Path(OUTPUT_GRASP) / f'dynamic_grasp_{condition}.csv'
    return pd.read_csv(path) if path.exists() else None


N_GRID = 300


def interp_trace(arr, n=N_GRID):
    x = np.linspace(0, 1, len(arr))
    return np.interp(np.linspace(0, 1, n), x, arr)


fig, axes = plt.subplots(1, 2)

for cond, color in COLORS_DG.items():
    df_dg = load_dynamic(cond)
    if df_dg is None or df_dg.empty:
        continue
    closed = df_dg[df_dg['phase'] != 'open']
    if closed.empty:
        continue
    T = np.linspace(0, len(closed) / 30, N_GRID)

    axes[0].plot(T, np.rad2deg(interp_trace(closed['q_7'].to_numpy())),
                 color=color, label=cond.capitalize())
    axes[1].plot(T, interp_trace(closed['k_tip_Npm'].to_numpy()),
                 color=color, label=cond.capitalize())

axes[0].set_xlabel('Time after close [s]')
axes[0].set_ylabel('Index MCP angle [deg]')
axes[0].legend()
axes[1].set_xlabel('Time after close [s]')
axes[1].set_ylabel('$K_{tip}$ [N/m]')
axes[1].legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_GRASP, 'dynamic_grasp_comparison.pdf'), bbox_inches='tight')
plt.show()